# NSE XBRL producer

Run manually from the financial_ingestion directory. The producer extracts source structure and publishes JSON. The consumer normalizes and saves it to MongoDB.


In [1]:
from pathlib import Path
import csv
import logging
import os
import uuid
import requests
import financial_ingestion
from financial_ingestion.settings import Settings
from financial_ingestion.broker import initialize_topology

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
settings = Settings.from_env()
if not settings.rabbit_url:
    raise ValueError("Set RABBITMQ_URL before running this notebook")
# Resolve the repository-root CSV through the editable local package.
CSV_PATH = Path(financial_ingestion.__file__).resolve().parents[2] / "symbols.csv"
with CSV_PATH.open(newline="", encoding="utf-8-sig") as file:
    reader = csv.DictReader(file)
    if not reader.fieldnames or "symbol" not in reader.fieldnames:
        raise ValueError("symbols.csv must contain a symbol column")
    symbols = list(dict.fromkeys(row["symbol"].strip().upper() for row in reader if row["symbol"].strip()))
if not symbols:
    raise ValueError("symbols.csv contains no symbols")
initialize_topology(settings)
run_id = str(uuid.uuid4())
print(f"Loaded {len(symbols)} symbols")


INFO Pika version 1.4.4 connecting to ('127.0.0.1', 5672)
ERROR Socket failed to connect: <socket.socket fd=632, family=2, type=1, proto=6, laddr=('0.0.0.0', 57577)>; error=10061 (Unknown error)
ERROR TCP Connection attempt failed: ConnectionRefusedError(10061, 'Unknown error'); dest=(<AddressFamily.AF_INET: 2>, <SocketKind.SOCK_STREAM: 1>, 6, '', ('127.0.0.1', 5672))
ERROR AMQPConnector - reporting failure: AMQPConnectorSocketConnectError: ConnectionRefusedError(10061, 'Unknown error')
ERROR AMQP connection workflow failed: AMQPConnectionWorkflowFailed: 1 exceptions in all; last exception - AMQPConnectorSocketConnectError: ConnectionRefusedError(10061, 'Unknown error'); first exception - None.
ERROR AMQPConnectionWorkflow - reporting failure: AMQPConnectionWorkflowFailed: 1 exceptions in all; last exception - AMQPConnectorSocketConnectError: ConnectionRefusedError(10061, 'Unknown error'); first exception - None
ERROR Connection workflow failed: AMQPConnectionWorkflowFailed: 1 exceptio

AMQPConnectionError: 

In [ ]:
from dataclasses import replace
from pathlib import Path
from urllib.parse import quote
from dotenv import dotenv_values
import financial_ingestion
import pika
from financial_ingestion.settings import Settings
from financial_ingestion.broker import initialize_topology

env_path = Path(financial_ingestion.__file__).resolve().parents[1] / ".env"
env = dotenv_values(env_path)

print("Config file:", env_path)
print("User present:", bool(env.get("RABBITMQ_USER")))
print("Password present:", bool(env.get("RABBITMQ_PASSWORD")))

user = env.get("RABBITMQ_USER")
password = env.get("RABBITMQ_PASSWORD")
if not user or not password:
    raise ValueError("Add RABBITMQ_USER and RABBITMQ_PASSWORD to the displayed .env file")

url = (
    f"amqp://{quote(user, safe='')}:{quote(password, safe='')}"
    "@127.0.0.1:5672/%2F"
)
settings = replace(Settings.from_env(), rabbit_url=url)

parsed = pika.URLParameters(settings.rabbit_url)
print("AMQP username:", parsed.credentials.username)
print("Password encoded and decoded correctly:", parsed.credentials.password == password)

initialize_topology(settings)
print("RabbitMQ topology initialized")

INFO Pika version 1.4.4 connecting to ('127.0.0.1', 5672)
INFO Socket connected: <socket.socket fd=1156, family=2, type=1, proto=6, laddr=('127.0.0.1', 56426), raddr=('127.0.0.1', 5672)>
INFO Streaming transport linked up: (<pika.adapters.utils.io_services_utils._AsyncPlaintextTransport object at 0x000001842A2642D0>, _StreamingProtocolShim: <SelectConnection PROTOCOL transport=<pika.adapters.utils.io_services_utils._AsyncPlaintextTransport object at 0x000001842A2642D0> params=<URLParameters host=127.0.0.1 port=5672 virtual_host=/ ssl=False>>).
INFO AMQPConnector - reporting success: <SelectConnection OPEN transport=<pika.adapters.utils.io_services_utils._AsyncPlaintextTransport object at 0x000001842A2642D0> params=<URLParameters host=127.0.0.1 port=5672 virtual_host=/ ssl=False>>
INFO AMQPConnectionWorkflow - reporting success: <SelectConnection OPEN transport=<pika.adapters.utils.io_services_utils._AsyncPlaintextTransport object at 0x000001842A2642D0> params=<URLParameters host=127.0.

Config file: C:\Repo\StockAnalysis_AI\financial_ingestion\.env
User present: True
Password present: True
AMQP username: financial_app
Password encoded and decoded correctly: True
RabbitMQ topology initialized


In [ ]:
from datetime import datetime, timezone
from urllib.parse import urlparse
from financial_ingestion.producers import fetch_nse_records, xbrl_produce
import uuid
run_id = str(uuid.uuid4())
# Issuer names come from PostgreSQL company_master; credentials belong in POSTGRES_DSN.
if not os.getenv("POSTGRES_DSN"):
    raise ValueError("Set POSTGRES_DSN for the XBRL issuer lookup")
import psycopg

def issuer_for(symbol):
    with psycopg.connect(os.environ["POSTGRES_DSN"]) as connection:
        with connection.cursor() as cursor:
            cursor.execute('SELECT "NAME_OF_COMPANY" FROM public.company_master WHERE "SYMBOL" = %s', (symbol,))
            row = cursor.fetchone()
    return row[0] if row else None

def selected(record, today=None):
    today = today or datetime.now(timezone.utc)
    period = str(record.get("period") or "")
    if period not in ("Annual", "Quarterly", "Half Yearly"): return False
    if record.get("audited") != "Audited" or record.get("consolidated") != "Consolidated": return False
    if record.get("format") != "New": return False
    if record.get("cumulative") != ("Cumulative" if period == "Annual" else "Non-cumulative"): return False
    url = urlparse(str(record.get("xbrl") or ""))
    if url.scheme != "https" or url.hostname not in ("nsearchives.nseindia.com", "archives.nseindia.com") or not url.path.lower().endswith(".xml"):
        return False
    try: start = datetime.strptime(record["fromDate"], "%d-%b-%Y")
    except (KeyError, ValueError): return False
    years = 5 if period == "Annual" else 2
    return start >= datetime(today.year - years, 1, 1)

success, failed = 0, 0
with requests.Session() as session:
    for symbol in symbols:
        try:
            issuer = issuer_for(symbol)
            if not issuer:
                logging.warning("No company_master issuer for %s", symbol)
                continue
            records = fetch_nse_records(session, symbol, issuer)
        except Exception:
            failed += 1
            logging.exception("XBRL metadata failed for %s; continuing", symbol)
            continue
        for record in records:
            if not selected(record): continue
            try:
                message = xbrl_produce(symbol, record, session, settings, correlation_id=run_id)
                success += 1
                logging.info("XBRL published %s filing=%s", symbol, record.get("seqNumber"))
            except Exception:
                failed += 1
                logging.exception("XBRL failed for %s filing=%s; continuing", symbol, record.get("seqNumber"))
print(f"XBRL: published={success} failed={failed}")


INFO Pika version 1.4.4 connecting to ('127.0.0.1', 5672)
ERROR Socket failed to connect: <socket.socket fd=1264, family=2, type=1, proto=6, laddr=('0.0.0.0', 57593)>; error=10061 (Unknown error)
ERROR TCP Connection attempt failed: ConnectionRefusedError(10061, 'Unknown error'); dest=(<AddressFamily.AF_INET: 2>, <SocketKind.SOCK_STREAM: 1>, 6, '', ('127.0.0.1', 5672))
ERROR AMQPConnector - reporting failure: AMQPConnectorSocketConnectError: ConnectionRefusedError(10061, 'Unknown error')
ERROR AMQP connection workflow failed: AMQPConnectionWorkflowFailed: 1 exceptions in all; last exception - AMQPConnectorSocketConnectError: ConnectionRefusedError(10061, 'Unknown error'); first exception - None.
ERROR AMQPConnectionWorkflow - reporting failure: AMQPConnectionWorkflowFailed: 1 exceptions in all; last exception - AMQPConnectorSocketConnectError: ConnectionRefusedError(10061, 'Unknown error'); first exception - None
ERROR Connection workflow failed: AMQPConnectionWorkflowFailed: 1 excepti